## Example 1: Basic ASGI application and uvicorn configuration

In [ ]:
import uvicorn
from contextlib import asynccontextmanager

# Example 1: Basic ASGI application and uvicorn configuration
print("Basic uvicorn ASGI server configuration:")

# Simple ASGI application
async def simple_app(scope, receive, send):
    """Basic ASGI application"""
    if scope['type'] == 'http':
        await send({
            'type': 'http.response.start',
            'status': 200,
            'headers': [[b'content-type', b'text/plain']],
        })
        await send({
            'type': 'http.response.body',
            'body': b'Hello from Uvicorn ASGI app!',
        })

# Create uvicorn configuration
config = uvicorn.Config(
    app=simple_app,
    host="127.0.0.1",
    port=8000,
    log_level="info",
    access_log=True
)

print(f"Uvicorn version: {uvicorn.__version__}")
print(f"Config created: {type(config).__name__}")
print(f"Host: {config.host}, Port: {config.port}")
print(f"Log level: {config.log_level}")

# Show configuration options
print(f"\nAvailable configuration options:")
config_attrs = ['host', 'port', 'log_level', 'access_log', 'reload', 'workers']
for attr in config_attrs:
    if hasattr(config, attr):
        print(f"  {attr}: {getattr(config, attr)}")

print("Uvicorn provides production-ready ASGI server capabilities")

## Example 2: FastAPI-style ASGI application with uvicorn

In [ ]:
import uvicorn
import asyncio
import json

# Example 2: FastAPI-style ASGI application with uvicorn
print("FastAPI-style ASGI application with uvicorn:")

class SimpleASGIFramework:
    """Simple ASGI framework for demonstration"""
    
    def __init__(self):
        self.routes = {}
    
    def route(self, path):
        """Route decorator"""
        def decorator(func):
            self.routes[path] = func
            return func
        return decorator
    
    async def __call__(self, scope, receive, send):
        """ASGI application callable"""
        if scope['type'] == 'http':
            path = scope['path']
            method = scope['method']
            
            # Route handling
            if path in self.routes:
                response_data = await self.routes[path](scope)
                response_body = json.dumps(response_data).encode()
                
                await send({
                    'type': 'http.response.start',
                    'status': 200,
                    'headers': [[b'content-type', b'application/json']],
                })
                await send({
                    'type': 'http.response.body',
                    'body': response_body,
                })
            else:
                # 404 response
                await send({
                    'type': 'http.response.start',
                    'status': 404,
                    'headers': [[b'content-type', b'text/plain']],
                })
                await send({
                    'type': 'http.response.body',
                    'body': b'Not Found',
                })

# Create app instance
app = SimpleASGIFramework()

@app.route("/")
async def home(scope):
    return {"message": "Hello from custom ASGI framework!", "path": scope['path']}

@app.route("/health")
async def health_check(scope):
    return {"status": "healthy", "version": "1.0.0"}

# Test different server configurations
configs = [
    {"host": "127.0.0.1", "port": 8000, "log_level": "info"},
    {"host": "0.0.0.0", "port": 8080, "log_level": "debug", "access_log": False},
    {"host": "127.0.0.1", "port": 9000, "log_level": "warning", "reload": False}
]

print("Different uvicorn configurations:")
for i, config_dict in enumerate(configs, 1):
    config = uvicorn.Config(app=app, **config_dict)
    print(f"Config {i}: {config_dict}")
    print(f"  Configured server: {config.host}:{config.port}")

print(f"\nASGI application configured with {len(app.routes)} routes")
print("Ready for production deployment with uvicorn")

## Example 3: Advanced server configuration and middleware

In [ ]:
import uvicorn
import asyncio
import logging

# Example 3: Advanced server configuration and middleware
print("Advanced uvicorn server configuration:")

# Custom logging configuration
logging_config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "default": {
            "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        },
    },
    "handlers": {
        "default": {
            "formatter": "default",
            "class": "logging.StreamHandler",
            "stream": "ext://sys.stdout",
        },
    },
    "root": {
        "level": "INFO",
        "handlers": ["default"],
    },
}

# ASGI application with middleware simulation
class MiddlewareASGIApp:
    """ASGI app with middleware layers"""
    
    def __init__(self):
        self.request_count = 0
    
    async def __call__(self, scope, receive, send):
        if scope['type'] == 'http':
            # Increment request counter
            self.request_count += 1
            
            # Simulate middleware processing
            start_time = asyncio.get_event_loop().time()
            
            # Process request
            path = scope.get('path', '/')
            method = scope.get('method', 'GET')
            
            response_data = {
                "method": method,
                "path": path,
                "request_number": self.request_count,
                "server": "uvicorn-demo"
            }
            
            response_body = str(response_data).encode()
            
            await send({
                'type': 'http.response.start',
                'status': 200,
                'headers': [
                    [b'content-type', b'text/plain'],
                    [b'x-request-id', str(self.request_count).encode()],
                    [b'x-server', b'uvicorn-demo'],
                ],
            })
            await send({
                'type': 'http.response.body',
                'body': response_body,
            })
            
            # Log processing time
            end_time = asyncio.get_event_loop().time()
            processing_time = end_time - start_time
            print(f"Request {self.request_count} processed in {processing_time:.4f}s")

# Create app
middleware_app = MiddlewareASGIApp()

# Advanced configuration options
advanced_configs = [
    {
        "name": "Development",
        "config": {
            "host": "127.0.0.1",
            "port": 8000,
            "log_level": "debug",
            "reload": True,
            "access_log": True
        }
    },
    {
        "name": "Production",
        "config": {
            "host": "0.0.0.0",
            "port": 80,
            "log_level": "info",
            "access_log": True,
            "workers": 4
        }
    },
    {
        "name": "High Performance",
        "config": {
            "host": "127.0.0.1",
            "port": 8080,
            "log_level": "warning",
            "access_log": False,
            "loop": "uvloop"  # High-performance event loop
        }
    }
]

print(f"\nAdvanced configuration examples:")
for config_info in advanced_configs:
    name = config_info["name"]
    config_dict = config_info["config"]
    
    # Create configuration (but don't start server)
    try:
        config = uvicorn.Config(app=middleware_app, **config_dict)
        print(f"{name} config:")
        print(f"  Host: {config.host}:{config.port}")
        print(f"  Log level: {config.log_level}")
        print(f"  Workers: {getattr(config, 'workers', 1)}")
    except Exception as e:
        print(f"{name} config: {e}")
        raise

print(f"\nMiddleware app ready with request tracking")
print("Uvicorn supports production deployments with multiple workers")

## Example 4: Server lifecycle management and graceful shutdown

In [ ]:
import uvicorn
import asyncio
import signal
import threading
import time

# Example 4: Server lifecycle management and graceful shutdown
print("Server lifecycle management with uvicorn:")

class ManagedASGIApp:
    """ASGI application with lifecycle management"""
    
    def __init__(self):
        self.startup_time = None
        self.request_count = 0
        self.is_running = False
    
    async def startup(self):
        """Simulate application startup"""
        self.startup_time = time.time()
        self.is_running = True
        print(f"Application started at {self.startup_time}")
    
    async def shutdown(self):
        """Simulate application shutdown"""
        self.is_running = False
        uptime = time.time() - self.startup_time if self.startup_time else 0
        print(f"Application shutting down after {uptime:.2f} seconds")
        print(f"Total requests served: {self.request_count}")
    
    async def __call__(self, scope, receive, send):
        if scope['type'] == 'lifespan':
            # Handle ASGI lifespan events
            message = await receive()
            if message['type'] == 'lifespan.startup':
                await self.startup()
                await send({'type': 'lifespan.startup.complete'})
            elif message['type'] == 'lifespan.shutdown':
                await self.shutdown()
                await send({'type': 'lifespan.shutdown.complete'})
                
        elif scope['type'] == 'http':
            if not self.is_running:
                # Server not ready
                await send({
                    'type': 'http.response.start',
                    'status': 503,
                    'headers': [[b'content-type', b'text/plain']],
                })
                await send({
                    'type': 'http.response.body',
                    'body': b'Service Unavailable',
                })
                return
            
            self.request_count += 1
            
            # Health check endpoint
            if scope['path'] == '/health':
                uptime = time.time() - self.startup_time
                response_data = {
                    "status": "healthy",
                    "uptime": f"{uptime:.2f}s",
                    "requests_served": self.request_count
                }
                response_body = str(response_data).encode()
            else:
                response_body = f"Hello! Request #{self.request_count}".encode()
            
            await send({
                'type': 'http.response.start',
                'status': 200,
                'headers': [[b'content-type', b'text/plain']],
            })
            await send({
                'type': 'http.response.body',
                'body': response_body,
            })

# Create managed app
managed_app = ManagedASGIApp()

# Test server lifecycle configuration
lifecycle_config = uvicorn.Config(
    app=managed_app,
    host="127.0.0.1",
    port=8000,
    log_level="info",
    lifespan="on"  # Enable lifespan events
)

print(f"Lifecycle configuration:")
print(f"  Host: {lifecycle_config.host}:{lifecycle_config.port}")
print(f"  Lifespan events: enabled")

# Simulate server management
class ServerManager:
    """Simulate server management operations"""
    
    def __init__(self, config):
        self.config = config
        self.server = None
    
    async def start_server(self):
        """Start the server (simulation)"""
        self.server = uvicorn.Server(self.config)
        print("Server instance created and ready to start")
        # In real usage: await self.server.serve()
        
    def stop_server(self):
        """Stop the server (simulation)"""
        if self.server:
            print("Server shutdown initiated")
            # In real usage: self.server.should_exit = True

# Demonstrate server management
manager = ServerManager(lifecycle_config)

print(f"\nServer management simulation:")
print("1. Creating server instance...")
# Note: We simulate this because actually starting a server would block the notebook
print("2. Server would start and handle lifespan events")
print("3. Application startup() called")
print("4. Server ready to accept requests")
print("5. On shutdown signal, shutdown() called")

print(f"\nProduction deployment considerations:")
print("- Graceful shutdown with signal handling")
print("- Lifespan event management")
print("- Health check endpoints")
print("- Request counting and monitoring")

## Example 5: Production deployment patterns and security

In [ ]:
import uvicorn
import asyncio
import os
import ssl

# Example 5: Production deployment patterns and security
print("Production deployment patterns with uvicorn:")

class ProductionASGIApp:
    """Production-ready ASGI application"""
    
    def __init__(self):
        self.metrics = {
            "requests": 0,
            "errors": 0,
            "total_response_time": 0.0
        }
        self.environment = os.getenv("ENVIRONMENT", "development")
    
    async def __call__(self, scope, receive, send):
        if scope['type'] == 'http':
            start_time = asyncio.get_event_loop().time()
            self.metrics["requests"] += 1
            
            try:
                # Handle different endpoints
                path = scope['path']
                method = scope['method']
                
                if path == '/metrics':
                    # Metrics endpoint for monitoring
                    avg_response_time = (
                        self.metrics["total_response_time"] / max(self.metrics["requests"], 1)
                    )
                    response_data = {
                        "requests_total": self.metrics["requests"],
                        "errors_total": self.metrics["errors"],
                        "avg_response_time_ms": round(avg_response_time * 1000, 2),
                        "environment": self.environment
                    }
                    response_body = str(response_data).encode()
                    
                elif path == '/health':
                    # Health check endpoint
                    response_body = b'{"status": "healthy"}'
                    
                elif path == '/ready':
                    # Readiness probe
                    response_body = b'{"status": "ready"}'
                    
                else:
                    # Default response
                    response_body = f"Production API - {method} {path}".encode()
                
                # Send response
                await send({
                    'type': 'http.response.start',
                    'status': 200,
                    'headers': [
                        [b'content-type', b'application/json' if path in ['/metrics', '/health', '/ready'] else b'text/plain'],
                        [b'x-environment', self.environment.encode()],
                        [b'x-request-id', str(self.metrics["requests"]).encode()],
                    ],
                })
                await send({
                    'type': 'http.response.body',
                    'body': response_body,
                })
                
            except Exception as e:
                self.metrics["errors"] += 1
                # Error response
                await send({
                    'type': 'http.response.start',
                    'status': 500,
                    'headers': [[b'content-type', b'text/plain']],
                })
                await send({
                    'type': 'http.response.body',
                    'body': b'Internal Server Error',
                })
            
            finally:
                # Record response time
                end_time = asyncio.get_event_loop().time()
                response_time = end_time - start_time
                self.metrics["total_response_time"] += response_time

# Production deployment configurations
deployment_configs = {
    "development": {
        "host": "127.0.0.1",
        "port": 8000,
        "log_level": "debug",
        "reload": True,
        "access_log": True
    },
    "staging": {
        "host": "0.0.0.0",
        "port": 8080,
        "log_level": "info",
        "access_log": True,
        "workers": 2
    },
    "production": {
        "host": "0.0.0.0",
        "port": 443,
        "log_level": "warning",
        "access_log": True,
        "workers": 4,
        # SSL configuration (would need actual cert files)
        # "ssl_keyfile": "/path/to/private.key",
        # "ssl_certfile": "/path/to/certificate.crt",
    }
}

print("Production deployment configurations:")

for env, config in deployment_configs.items():
    print(f"\n{env.upper()} environment:")
    
    # Create app for this environment
    os.environ["ENVIRONMENT"] = env
    app = ProductionASGIApp()
    
    try:
        # Create uvicorn config
        uvicorn_config = uvicorn.Config(app=app, **config)
        
        print(f"  Host: {uvicorn_config.host}:{uvicorn_config.port}")
        print(f"  Log level: {uvicorn_config.log_level}")
        print(f"  Workers: {getattr(uvicorn_config, 'workers', 1)}")
        print(f"  Reload: {getattr(uvicorn_config, 'reload', False)}")
        print(f"  SSL: {'Yes' if hasattr(uvicorn_config, 'ssl_keyfile') else 'No'}")
        
    except Exception as e:
        print(f"  Configuration error: {e}")
        raise

# Security considerations
print(f"\nSecurity and monitoring features:")
security_features = [
    "SSL/TLS termination",
    "Request ID tracking",
    "Environment-specific configuration",
    "Health and readiness probes",
    "Metrics collection",
    "Error tracking",
    "Access logging",
    "Worker process isolation"
]

for feature in security_features:
    print(f"  ✓ {feature}")

print(f"\nDeployment command examples:")
print("Development: uvicorn app:app --host 127.0.0.1 --port 8000 --reload")
print("Production:  uvicorn app:app --host 0.0.0.0 --port 80 --workers 4")
print("With SSL:    uvicorn app:app --host 0.0.0.0 --port 443 --ssl-keyfile key.pem --ssl-certfile cert.pem")

print(f"\nUvicorn provides enterprise-grade ASGI server capabilities")